# Practical Exam: Spectrum Shades LLC
Spectrum Shades LLC is a prominent supplier of concrete color solutions, offering a wide range of pigments and coloring systems used in various concrete applications, including decorative concrete, precast concrete, and concrete pavers. The company prides itself on delivering high-quality colorants that meet the unique needs of its diverse clientele, including contractors, architects, and construction companies.
</br></br>
The company has recently observed a growing number of customer complaints regarding inconsistent color quality in their products. The discrepancies have led to a decline in customer satisfaction and a potential increase in product returns.
By identifying and mitigating the factors causing color variations, the company can enhance product reliability, reduce customer complaints, and minimize return rates.
</br></br>
You are part of the data analysis team tasked with providing actionable insights to help Spectrum Shades LLC address the issues of inconsistent color quality and improve customer satisfaction.

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `production_data` table, and the data team have provided the following data description. 

Write a query to ensure the data matches the description provided, including identifying and cleaning all invalid values. You must match all column names and description criteria.
</br>

- You should start with the data in the file "production_data.csv".
- Your output should be a DataFrame named clean_data.
- All column names and values should match the table below.
</br>

| Column Name             | Criteria                                                                                         |
|--------------------------|--------------------------------------------------------------------------------------------------|
| batch_id | Discrete. Identifier for each batch. Missing values are not possible. |
| production_date | Date. Date when the batch was produced.|
| raw_material_supplier | Categorical. Supplier of the raw materials. (1='national_supplier', 2='international_supplier'). <br> Missing values should be replaced with 'national_supplier'.|
| pigment_type           | Nominal. Type of pigment used. ['type_a', 'type_b', 'type_c']. <br> Missing values should be replaced with 'other'. |
| pigment_quantity       | Continuous. Amount of pigment added (in kilograms) (Range: 1 - 100). <br> Missing values should be replaced with median. |
| mixing_time           | Continuous. Duration of the mixing process (in minutes). <br> Missing values should be replaced with mean, rounded to 2 decimal places. |
| mixing_speed          | Categorical. Speed of the mixing process represented as categories: 'Low', 'Medium', 'High'.</br> Missing values should be replaced with 'Not Specified'. |
| product_quality_score | Continuous. Overall quality score of the final product (rating on a scale of 1 to 10). <br> Missing values should be replaced with mean, rounded to 2 decimal places. |


In [9]:
import pandas as pd

# Load dataset
data = pd.read_csv('production_data.csv')

# -------------------------------
# Convert to correct data types
# -------------------------------
data['batch_id'] = pd.to_numeric(data['batch_id'], errors='coerce').astype('Int64')
data['pigment_quantity'] = pd.to_numeric(data['pigment_quantity'], errors='coerce')
data['mixing_time'] = pd.to_numeric(data['mixing_time'], errors='coerce')
data['product_quality_score'] = pd.to_numeric(data['product_quality_score'], errors='coerce')

# -------------------------------
# Drop missing/invalid batch_id
# -------------------------------
data = data.dropna(subset=['batch_id'])

# -------------------------------
# production_date → must be a valid datetime64
# -------------------------------
data['production_date'] = pd.to_datetime(data['production_date'], errors='coerce')
data = data.dropna(subset=['production_date'])

# -------------------------------
# raw_material_supplier → clean strings, then map
# -------------------------------
data['raw_material_supplier'] = data['raw_material_supplier'].astype(str).str.strip().str.lower()
data['raw_material_supplier'] = data['raw_material_supplier'].replace({
    '1': 'national_supplier',
    '2': 'international_supplier',
    'national_supplier': 'national_supplier',
    'international_supplier': 'international_supplier',
    'nan': 'national_supplier'   # catch string "nan"
})
data['raw_material_supplier'] = data['raw_material_supplier'].fillna('national_supplier')

# -------------------------------
# pigment_type → clean text before validation
# -------------------------------
data['pigment_type'] = data['pigment_type'].astype(str).str.strip().str.lower()
valid_pigments = ['type_a', 'type_b', 'type_c']
data['pigment_type'] = data['pigment_type'].apply(lambda x: x if x in valid_pigments else 'other')

# -------------------------------
# pigment_quantity → numeric, must be in [1,100], fill invalid/missing with median
# -------------------------------
median_pigment = data['pigment_quantity'].median()
data['pigment_quantity'] = data['pigment_quantity'].apply(
    lambda x: median_pigment if pd.isna(x) or x < 1 or x > 100 else x
)

# -------------------------------
# mixing_time → numeric, fill missing with mean (rounded to 2 decimals)
# -------------------------------
mean_mixing_time = round(data['mixing_time'].mean(), 2)
data['mixing_time'] = data['mixing_time'].fillna(mean_mixing_time)

# -------------------------------
# mixing_speed → clean text before validation
# -------------------------------
data['mixing_speed'] = data['mixing_speed'].astype(str).str.strip().str.lower()
data['mixing_speed'] = data['mixing_speed'].str.capitalize()
valid_speeds = ['Low', 'Medium', 'High']
data['mixing_speed'] = data['mixing_speed'].apply(
    lambda x: x if x in valid_speeds else 'Not Specified'
)

# -------------------------------
# product_quality_score → numeric, must be in [1,10], fill invalid/missing with mean
# -------------------------------
mean_quality_score = round(data['product_quality_score'].mean(), 2)
data['product_quality_score'] = data['product_quality_score'].apply(
    lambda x: mean_quality_score if pd.isna(x) or x < 1 or x > 10 else x
)

# -------------------------------
# Final formatting
# -------------------------------
# Keep production_date as datetime64 (don’t convert to string)
# Round continuous variables to 2 decimal places
data['pigment_quantity'] = data['pigment_quantity'].round(2)
data['mixing_time'] = data['mixing_time'].round(2)
data['product_quality_score'] = data['product_quality_score'].round(2)

# -------------------------------
# Validation checks (assertions)
# -------------------------------
assert data['batch_id'].notna().all()
assert pd.api.types.is_datetime64_any_dtype(data['production_date'])
assert data['raw_material_supplier'].isin(['national_supplier','international_supplier']).all()
assert data['pigment_type'].isin(['type_a','type_b','type_c','other']).all()
assert data['pigment_quantity'].between(1, 100).all()
assert data['mixing_speed'].isin(['Low','Medium','High','Not Specified']).all()
assert data['product_quality_score'].between(1, 10).all()

# -------------------------------
# Final cleaned DataFrame
# -------------------------------
clean_data = data.copy()
clean_data.head()
clean_data.dtypes
clean_data.isna().sum()



batch_id                 0
production_date          0
raw_material_supplier    0
pigment_type             0
pigment_quantity         0
mixing_time              0
mixing_speed             0
product_quality_score    0
dtype: int64

# Task 2

You want to understand how the supplier type and quantity of materials affect the final product attributes.

Calculate the average `product_quality_score` and `pigment_quantity` grouped by `raw_material_supplier`.

- You should start with the data in the file 'production_data.csv'. 
- Your output should be a DataFrame named aggregated_data.
- It should include the three columns: `raw_material_supplier`, `avg_product_quality_score`, and `avg_pigment_quantity`.
- Your answers should be rounded to 2 decimal places.


In [10]:
# Write your answer to Task 2 here
import pandas as pd
import numpy as np

# Load the CSV file
data = pd.read_csv('production_data.csv')

# --- Step 1: Clean the relevant columns (only what's needed for this task) ---

# raw_material_supplier: map 1/2 and fill missing
data['raw_material_supplier'] = data['raw_material_supplier'].replace({1: 'national_supplier', 
                                                                     2: 'international_supplier'})
data['raw_material_supplier'] = data['raw_material_supplier'].fillna('national_supplier')

# pigment_quantity: replace missing with median
median_pigment = data['pigment_quantity'].median()
data['pigment_quantity'] = data['pigment_quantity'].apply(lambda x: median_pigment if pd.isna(x) else x)

# product_quality_score: replace missing with mean
mean_quality_score = data['product_quality_score'].mean()
data['product_quality_score'] = data['product_quality_score'].apply(lambda x: mean_quality_score if pd.isna(x) else x)

# --- Step 2: Group by supplier and calculate averages ---
aggregated_data = data.groupby('raw_material_supplier').agg(
    avg_product_quality_score=('product_quality_score', lambda x: round(x.mean(), 2)),
    avg_pigment_quantity=('pigment_quantity', lambda x: round(x.mean(), 2))
).reset_index()

# Display the result
aggregated_data

,raw_material_supplier,avg_product_quality_score,avg_pigment_quantity
0,international_supplier,5.97,34.91
1,national_supplier,8.02,44.73


# Task 3

To get more insight into the factors behind product quality, you want to filter the data to see an average product quality score for a specified set of results.

Identify the average `product_quality_score` for batches with a `raw_material_supplier` of 2 and a `pigment_quantity` greater than 35 kg.

Write a query to return the average `avg_product_quality_score` for these filtered batches. Use the original production data table, not the output of Task 2.

- You should start with the data in the file 'production_data.csv'. 
- Your output should be a DataFrame named pigment_data.
- It should consist of a 1-row DataFrame with 3 columns: `raw_material_supplier`, `pigment_quantity`, and `avg_product_quality_score`.
- Your answers should be rounded to 2 decimal places where appropriate.


In [11]:
# Write your answer to Task 3 here
import pandas as pd

data = pd.read_csv('production_data.csv')

# Filter: supplier = 2, pigment_quantity > 35
filtered = data[(data['raw_material_supplier'] == 2) & (data['pigment_quantity'] > 35)]

# Compute averages (rounded)
avg_quality = round(filtered['product_quality_score'].mean(), 2)
avg_pigment = round(filtered['pigment_quantity'].mean(), 2)

# Output DataFrame
pigment_data = pd.DataFrame({
    'raw_material_supplier': [2],
    'pigment_quantity': [avg_pigment],
    'avg_product_quality_score': [avg_quality]
})

pigment_data

,raw_material_supplier,pigment_quantity,avg_product_quality_score
0,2,39.01,5.97


# Task 4

In order to proceed with further analysis later, you need to analyze how various factors relate to product quality. Start by calculating the mean and standard deviation for the following columns: `pigment_quantity`, and `product_quality_score`. </br> These statistics will help in understanding the central tendency and variability of the data related to product quality.
</br> </br >
Next, calculate the Pearson correlation coefficient between the following variables: `pigment_quantity`, and `product_quality_score`.
</br>
These correlation coefficients will provide insights into the strength and direction of the relationships between the factors and overall product quality.


- You should start with the data in the file 'production_data.csv'.
- Calculate the mean and standard deviation for the columns pigment_quantity and product_quality_score as: `product_quality_score_mean`, `product_quality_score_sd`, `pigment_quantity_mean`, `pigment_quantity_sd`.
- Calculate the Pearson correlation coefficient between pigment_quantity and product_quality_score as: `corr_coef`
- Your output should be a DataFrame named product_quality.
- It should include the columns: `product_quality_score_mean`, `product_quality_score_sd`, `pigment_quantity_mean`, `pigment_quantity_sd`, `corr_coef`.
- Ensure that your answers are rounded to 2 decimal places.


In [12]:
# Write your answer to Task 4 here
import pandas as pd

data = pd.read_csv('production_data.csv')

median_pigment = data['pigment_quantity'].median()
data['pigment_quantity'] = data['pigment_quantity'].fillna(median_pigment)

mean_quality_score = data['product_quality_score'].median()
data['product_quality_score'] = data['product_quality_score'].fillna(mean_quality_score)

product_quality_score_mean = round(data['product_quality_score'].mean(), 2)
product_quality_score_sd = round(data['product_quality_score'].std(), 2)

pigment_quantity_mean = round(data['pigment_quantity'].mean(), 2)
pigment_quantity_sd = round(data['pigment_quantity'].std(), 2)

corr_coef = round(data['pigment_quantity'].corr(data['product_quality_score']), 2)

product_quality = pd.DataFrame({
    'product_quality_score_mean':[product_quality_score_mean],
    'product_quality_score_sd':[product_quality_score_sd],
    'pigment_quantity_mean':[pigment_quantity_mean],
    'pigment_quantity_sd': [pigment_quantity_sd],
    'corr_coef': [corr_coef]
})
product_quality

,product_quality_score_mean,product_quality_score_sd,pigment_quantity_mean,pigment_quantity_sd,corr_coef
0,6.68,1.39,38.35,6.83,0.49


# FORMATTING AND NAMING CHECK
Use the code block below to check that your outputs are correctly named and formatted before you submit your project.

This code checks whether you have met our automarking requirements: that the specified DataFrames exist and contain the required columns. It then prints a table showing ✅ for each column that exists and ❌ for any that are missing, or if the DataFrame itself isn't available.

If a DataFrame or a column in a DataFrame doesn't exist, carefully check your code again.

IMPORTANT: even if your code passes the check below, this does not mean that your entire submission is correct. This is a check for naming and formatting only.

In [13]:
import pandas as pd

def check_columns(output_df, output_df_name, required_columns):
    results = []
    for col in required_columns:
        exists = col in output_df.columns
        results.append({'Dataset': output_df_name, 'Column': col, 'Exists': '✅' if exists else '❌'})
    return results

def safe_check(output_df_name, required_columns):
    results = []
    if output_df_name in globals():
        obj = globals()[output_df_name]
        if isinstance(obj, pd.DataFrame):
            results.extend(check_columns(obj, output_df_name, required_columns))
        elif isinstance(obj, str) and ("SELECT" in obj.upper() or "FROM" in obj.upper()):
            results.append({'Dataset': output_df_name, 'Column': '—', 'Exists': 'ℹ️ SQL query string'})
        else:
            results.append({'Dataset': output_df_name, 'Column': '—', 'Exists': '❌ Not a DataFrame or query'})
    else:
        results.append({'Dataset': output_df_name, 'Column': '—', 'Exists': '❌ Variable not defined'})
    return results

requirements = {
    'clean_data': ['production_date', 'pigment_type', 'mixing_time', 'mixing_speed'],
    'aggregated_data': ['raw_material_supplier', 'avg_product_quality_score', 'avg_pigment_quantity'],
    'pigment_data': ['raw_material_supplier', 'pigment_quantity', 'avg_product_quality_score'],
    'product_quality': ['product_quality_score_mean', 'product_quality_score_sd',
                        'pigment_quantity_mean', 'pigment_quantity_sd', 'corr_coef']
}

all_results = []
for output_df_name, cols in requirements.items():
    all_results += safe_check(output_df_name, cols)

check_results_df = pd.DataFrame(all_results)

print(check_results_df)

            Dataset                      Column Exists
0        clean_data             production_date      ✅
1        clean_data                pigment_type      ✅
2        clean_data                 mixing_time      ✅
3        clean_data                mixing_speed      ✅
4   aggregated_data       raw_material_supplier      ✅
5   aggregated_data   avg_product_quality_score      ✅
6   aggregated_data        avg_pigment_quantity      ✅
7      pigment_data       raw_material_supplier      ✅
8      pigment_data            pigment_quantity      ✅
9      pigment_data   avg_product_quality_score      ✅
10  product_quality  product_quality_score_mean      ✅
11  product_quality    product_quality_score_sd      ✅
12  product_quality       pigment_quantity_mean      ✅
13  product_quality         pigment_quantity_sd      ✅
14  product_quality                   corr_coef      ✅
